In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import yfinance as yf
import seaborn as sns

import sys
sys.path.append('../utils')   # path relative to the notebook
from common import fetch_portfolio_data, market_cap

In [4]:
# 1) Load S&P 500 constituents with sectors
url = "https://datahub.io/core/s-and-p-500-companies/_r/-/data/constituents.csv"
df = pd.read_csv(url)
display(df.head())
display(df.tail())

,Symbol,Security,GICS Sector,GICS Sub-Industry,Headquarters Location,Date added,CIK,Founded
0,MMM,3M,Industrials,Industrial Conglomerates,"Saint Paul, Minnesota",1957-03-04,66740,1902
1,AOS,A. O. Smith,Industrials,Building Products,"Milwaukee, Wisconsin",2017-07-26,91142,1916
2,ABT,Abbott Laboratories,Health Care,Health Care Equipment,"North Chicago, Illinois",1957-03-04,1800,1888
3,ABBV,AbbVie,Health Care,Biotechnology,"North Chicago, Illinois",2012-12-31,1551152,2013 (1888)
4,ACN,Accenture,Information Technology,IT Consulting & Other Services,"Dublin, Ireland",2011-07-06,1467373,1989


,Symbol,Security,GICS Sector,GICS Sub-Industry,Headquarters Location,Date added,CIK,Founded
498,XYL,Xylem Inc.,Industrials,Industrial Machinery & Supplies & Components,"White Plains, New York",2011-11-01,1524472,2011
499,YUM,Yum! Brands,Consumer Discretionary,Restaurants,"Louisville, Kentucky",1997-10-06,1041061,1997
500,ZBRA,Zebra Technologies,Information Technology,Electronic Equipment & Instruments,"Lincolnshire, Illinois",2019-12-23,877212,1969
501,ZBH,Zimmer Biomet,Health Care,Health Care Equipment,"Warsaw, Indiana",2001-08-07,1136869,1927
502,ZTS,Zoetis,Health Care,Pharmaceuticals,"Parsippany, New Jersey",2013-06-21,1555280,1952


In [5]:
# 2) Get unique tickers and fetch market caps in one call
tickers = df["Symbol"].unique().tolist()
caps_df = market_cap(tickers)  # {ticker: cap}

In [22]:
# 3) Extract dict and map
caps_dict = caps_df['MarketCap'].to_dict()
df["MarketCap"] = df["Symbol"].map(caps_dict)

print(f"✅ Mapped {len(caps_dict)} tickers, {df['MarketCap'].isna().sum()} missing")
df = df.dropna(subset=["MarketCap"])  # Clean slate


✅ Mapped 503 tickers, 0 missing


/var/folders/89/hrgs9sm57xz4lcmzgjzdswph0000gn/T/ipykernel_93447/1812029764.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["MarketCap"] = df["Symbol"].map(caps_dict)


In [77]:
# 4) Print tickers and caps grouped by sector
grouped = df.groupby("GICS Sector")

for sector, sub in grouped:
    print(f"\n========= {sector} =========")
    # sort by ticker or cap as you like
    sub_sorted = sub.sort_values("Symbol")
    for _, row in sub_sorted.iterrows():
        print(f"{row['Symbol']}, ${row['MarketCap']/1e6:,.0f} M")


========= Communication Services =========
CHTR, $26,974 M
CMCSA, $105,746 M
DIS, $176,470 M
EA, $50,178 M
FOX, $24,360 M
FOXA, $24,596 M
GOOG, $3,614,463 M
GOOGL, $3,641,197 M
LYV, $34,950 M
META, $1,501,696 M
MTCH, $7,332 M
NFLX, $389,490 M
NWS, $15,103 M
NWSA, $13,815 M
OMC, $23,614 M
PSKY, $10,172 M
T, $200,702 M
TKO, $14,867 M
TMUS, $233,175 M
TTD, $11,476 M
TTWO, $37,152 M
VZ, $210,800 M
WBD, $68,029 M

========= Consumer Discretionary =========
ABNB, $77,050 M
AMZN, $2,204,631 M
APTV, $14,515 M
AZO, $54,230 M
BBY, $13,194 M
BKNG, $139,380 M
CCL, $33,410 M
CMG, $44,124 M
CVNA, $61,640 M
DASH, $68,049 M
DECK, $14,637 M
DHI, $38,751 M
DPZ, $12,555 M
DRI, $23,618 M
EBAY, $40,219 M
EXPE, $28,818 M
F, $45,959 M
GM, $67,922 M
GPC, $13,409 M
GRMN, $44,725 M
HAS, $12,818 M
HD, $319,310 M
HLT, $68,006 M
LEN, $22,422 M
LOW, $126,007 M
LULU, $19,101 M
LVS, $35,788 M
MAR, $84,731 M
MCD, $220,397 M
MGM, $9,674 M
NCLH, $8,633 M
NKE, $77,528 M
NVR, $18,085 M
ORLY, $73,690 M
PHM, $22,010 M
POOL

In [66]:
# 5) Compute mean market cap (overall and by sector)
overall_median = df["MarketCap"].median()
print("\n========= Overall =========")
print(f"Median market cap across S&P 500: ${overall_median/1e6:,.0f} M")

sector_means = grouped["MarketCap"].median().sort_values()

print("\n========= Sector mean market caps =========")
for sector, mcap in sector_means.items():
    print(f"{sector}: ${mcap/1e6:,.0f} M")


========= Overall =========
Median market cap across S&P 500: $39,279 M

========= Sector mean market caps =========
Real Estate: $22,736 M
Materials: $23,779 M
Utilities: $30,964 M
Consumer Staples: $31,841 M
Health Care: $32,384 M
Communication Services: $37,152 M
Consumer Discretionary: $39,485 M
Financials: $40,780 M
Industrials: $41,435 M
Energy: $59,758 M
Information Technology: $60,533 M


In [101]:
# UNIVERSAL: 5 stocks closest to overall mean (any pandas version)
overall_median = df["MarketCap"].median()
print(f"\nOverall mean market cap: ${overall_median/1e6:,.0f} M")

def closest_to_median(group, target_median, n=5):
    group = group.copy()
    group["dist_to_median"] = abs(group["MarketCap"] - target_median)
    return group.nsmallest(n, "dist_to_median")[["Symbol", "MarketCap", "GICS Sector"]]

# Works everywhere: select cols first, then groupby.apply
closest_stocks = (
    df[["GICS Sector", "Symbol", "MarketCap"]]  # Explicit cols = no warning
    .groupby("GICS Sector", group_keys=False)
    .apply(closest_to_median, target_median=overall_median, n=1)
    .reset_index(drop=True)  # Clean flat DataFrame
    .round(0)
)

print("\n========= STOCKS CLOSEST TO MEAN MCAP PER SECTOR =========")

for sector in closest_stocks["GICS Sector"].unique():
    sector_df = closest_stocks[closest_stocks["GICS Sector"] == sector]
    print(f"\n====== {sector}: ======")
    for _, row in sector_df.iterrows():
        print(f"  {row['Symbol']:6}: ${row['MarketCap']/1e6:,.0f} M, Dev: {(row['MarketCap']-overall_median)/overall_median:,.3f}")



Overall mean market cap: $39,279 M

========= STOCKS CLOSEST TO MEAN MCAP PER SECTOR =========

====== Communication Services: ======
  TTWO  : $37,152 M, Dev: -0.054

====== Consumer Discretionary: ======
  DHI   : $38,751 M, Dev: -0.013

====== Consumer Staples: ======
  SYY   : $38,951 M, Dev: -0.008

====== Energy: ======
  EQT   : $40,372 M, Dev: 0.028

====== Financials: ======
  FITB  : $40,017 M, Dev: 0.019

====== Health Care: ======
  RMD   : $32,987 M, Dev: -0.160

====== Industrials: ======
  AXON  : $39,899 M, Dev: 0.016

====== Information Technology: ======
  ROP   : $38,069 M, Dev: -0.031

====== Materials: ======
  NUE   : $36,292 M, Dev: -0.076

====== Real Estate: ======
  CBRE  : $39,279 M, Dev: 0.000

====== Utilities: ======
  ED    : $39,545 M, Dev: 0.007


/var/folders/89/hrgs9sm57xz4lcmzgjzdswph0000gn/T/ipykernel_93447/1148015197.py:14: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(closest_to_median, target_median=overall_median, n=1)
